In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display

code_dir = Path.cwd().resolve()
if not (code_dir / "topic_nn_helpers.py").exists():
    if (code_dir / "code" / "topic_nn_helpers.py").exists():
        code_dir = code_dir / "code"
    elif (code_dir.parent / "code" / "topic_nn_helpers.py").exists():
        code_dir = code_dir.parent / "code"
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

from topic_nn_helpers import (
    ZERO_SHOT_MAJOR_TOPICS,
    build_tag_matrices,
    build_target_matrix,
    build_topk_frequency_baseline,
    choose_thresholds,
    dataset_summary_table,
    evaluate_multilabel_predictions,
    fit_logistic_baseline,
    group_train_val_test_split,
    load_review_topic_dataset,
    make_feature_set_summary,
    make_prediction_frame,
    predict_logistic_probabilities,
    predict_mlp_probabilities,
    run_grouped_logistic_cv,
    save_json,
    train_mlp,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

RANDOM_STATE = 42
MIN_TAG_DF = 5
ACTIVE_FEATURE_SET = "all_tags"
FEATURE_SPECS = {
    "all_tags": "Normalized_Tags",
    "structural_tags": "Structural_Tags",
    "room_tags": "Room_Tags",
}
RUN_GROUPED_CV = False
MLP_CONFIG = {
    "hidden_dims": (256, 128),
    "dropout": 0.2,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 256,
    "max_epochs": 60,
    "patience": 12,
    "min_delta": 1e-4,
    "random_state": RANDOM_STATE,
}
OUTPUT_DIR = code_dir / "BERTModelRawOutputs" / "guest_clusters_outputs" / "topic_nn_review_level"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR


In [ ]:
# Load the review-level tags and aggregate the fragment-level topic labels back to one row per review.
dataset = load_review_topic_dataset(code_dir)
dataset_summary = dataset_summary_table(dataset)

print(f"Rows with at least one zero-shot topic: {len(dataset):,}")
print(f"Hotels represented: {dataset['Hotel_Name'].nunique():,}")
display(dataset_summary)
display(
    dataset[
        [
            "Person_id",
            "Hotel_Name",
            "City",
            "Tag_Flag_Count",
            "Active_Topic_Count",
            *ZERO_SHOT_MAJOR_TOPICS,
        ]
    ].head()
)
display(dataset["Active_Topic_Count"].value_counts(normalize=True).rename("share").reset_index())


In [ ]:
# Split by hotel to avoid leaking hotel-specific tag/topic patterns across train and test.
splits, split_summary = group_train_val_test_split(dataset, random_state=RANDOM_STATE)
feature_summary = make_feature_set_summary(dataset, splits, FEATURE_SPECS, min_df=MIN_TAG_DF)
target_matrices = {split_name: build_target_matrix(split_df) for split_name, split_df in splits.items()}

hotel_overlap = pd.DataFrame(
    [
        {
            "Pair": "train_vs_validation",
            "Shared_Hotels": len(set(splits["train"]["Hotel_Name"]) & set(splits["validation"]["Hotel_Name"])),
        },
        {
            "Pair": "train_vs_test",
            "Shared_Hotels": len(set(splits["train"]["Hotel_Name"]) & set(splits["test"]["Hotel_Name"])),
        },
        {
            "Pair": "validation_vs_test",
            "Shared_Hotels": len(set(splits["validation"]["Hotel_Name"]) & set(splits["test"]["Hotel_Name"])),
        },
    ]
)

display(split_summary)
display(hotel_overlap)
display(feature_summary)


In [ ]:
# Run a logistic-regression ablation across the tag feature sets before training the MLP.
feature_artifacts = {}
ablation_rows = []
ablation_topic_rows = []

for feature_name, tag_column in FEATURE_SPECS.items():
    matrices, vocabulary_df = build_tag_matrices(
        splits["train"],
        splits["validation"],
        splits["test"],
        tag_column=tag_column,
        min_df=MIN_TAG_DF,
    )
    logistic_model = fit_logistic_baseline(matrices["train"], target_matrices["train"])
    validation_prob = predict_logistic_probabilities(logistic_model, matrices["validation"])
    thresholds = choose_thresholds(target_matrices["validation"], validation_prob)
    test_prob = predict_logistic_probabilities(logistic_model, matrices["test"])
    overall_metrics, per_topic_metrics = evaluate_multilabel_predictions(
        target_matrices["test"],
        test_prob,
        ZERO_SHOT_MAJOR_TOPICS,
        model_name=f"logistic_{feature_name}",
        split_name="test",
        thresholds=thresholds,
    )
    overall_metrics.insert(0, "Feature_Set", feature_name)
    overall_metrics.insert(1, "Tag_Column", tag_column)
    overall_metrics.insert(2, "Feature_Count", matrices["train"].shape[1])
    per_topic_metrics.insert(0, "Feature_Set", feature_name)
    ablation_rows.append(overall_metrics)
    ablation_topic_rows.append(per_topic_metrics)
    feature_artifacts[feature_name] = {
        "tag_column": tag_column,
        "matrices": matrices,
        "vocabulary": vocabulary_df,
        "model": logistic_model,
        "thresholds": thresholds,
    }

ablation_overall = pd.concat(ablation_rows, ignore_index=True).sort_values(["Macro_F1", "Micro_F1"], ascending=False)
ablation_per_topic = pd.concat(ablation_topic_rows, ignore_index=True)
active_features = feature_artifacts[ACTIVE_FEATURE_SET]
tag_vocabulary = active_features["vocabulary"].copy()

display(ablation_overall)
display(tag_vocabulary.head(20))


In [ ]:
# Train the baselines and the MLP on the active feature set.
active_matrices = active_features["matrices"]
y_train = target_matrices["train"]
y_validation = target_matrices["validation"]
y_test = target_matrices["test"]

frequency_validation_prob, frequency_validation_pred = build_topk_frequency_baseline(y_train, len(splits["validation"]))
frequency_test_prob, frequency_test_pred = build_topk_frequency_baseline(y_train, len(splits["test"]))

frequency_validation_metrics, frequency_validation_topic_metrics = evaluate_multilabel_predictions(
    y_validation,
    frequency_validation_prob,
    ZERO_SHOT_MAJOR_TOPICS,
    model_name="frequency_baseline",
    split_name="validation",
    predictions_override=frequency_validation_pred,
)
frequency_test_metrics, frequency_test_topic_metrics = evaluate_multilabel_predictions(
    y_test,
    frequency_test_prob,
    ZERO_SHOT_MAJOR_TOPICS,
    model_name="frequency_baseline",
    split_name="test",
    predictions_override=frequency_test_pred,
)

logistic_model = active_features["model"]
logistic_thresholds = active_features["thresholds"]
logistic_validation_prob = predict_logistic_probabilities(logistic_model, active_matrices["validation"])
logistic_test_prob = predict_logistic_probabilities(logistic_model, active_matrices["test"])
logistic_validation_metrics, logistic_validation_topic_metrics = evaluate_multilabel_predictions(
    y_validation,
    logistic_validation_prob,
    ZERO_SHOT_MAJOR_TOPICS,
    model_name="logistic_regression",
    split_name="validation",
    thresholds=logistic_thresholds,
)
logistic_test_metrics, logistic_test_topic_metrics = evaluate_multilabel_predictions(
    y_test,
    logistic_test_prob,
    ZERO_SHOT_MAJOR_TOPICS,
    model_name="logistic_regression",
    split_name="test",
    thresholds=logistic_thresholds,
)

mlp_model, mlp_history, mlp_config = train_mlp(
    active_matrices["train"],
    y_train,
    active_matrices["validation"],
    y_validation,
    **MLP_CONFIG,
)
mlp_validation_prob = predict_mlp_probabilities(mlp_model, active_matrices["validation"], device=mlp_config["device"])
mlp_thresholds = choose_thresholds(y_validation, mlp_validation_prob)
mlp_test_prob = predict_mlp_probabilities(mlp_model, active_matrices["test"], device=mlp_config["device"])
mlp_validation_metrics, mlp_validation_topic_metrics = evaluate_multilabel_predictions(
    y_validation,
    mlp_validation_prob,
    ZERO_SHOT_MAJOR_TOPICS,
    model_name="mlp",
    split_name="validation",
    thresholds=mlp_thresholds,
)
mlp_test_metrics, mlp_test_topic_metrics = evaluate_multilabel_predictions(
    y_test,
    mlp_test_prob,
    ZERO_SHOT_MAJOR_TOPICS,
    model_name="mlp",
    split_name="test",
    thresholds=mlp_thresholds,
)

overall_metrics = pd.concat(
    [
        frequency_validation_metrics,
        frequency_test_metrics,
        logistic_validation_metrics,
        logistic_test_metrics,
        mlp_validation_metrics,
        mlp_test_metrics,
    ],
    ignore_index=True,
)
per_topic_metrics = pd.concat(
    [
        frequency_validation_topic_metrics,
        frequency_test_topic_metrics,
        logistic_validation_topic_metrics,
        logistic_test_topic_metrics,
        mlp_validation_topic_metrics,
        mlp_test_topic_metrics,
    ],
    ignore_index=True,
)

prediction_frames = pd.concat(
    [
        make_prediction_frame(
            splits["validation"],
            y_validation,
            logistic_validation_prob,
            ZERO_SHOT_MAJOR_TOPICS,
            logistic_thresholds,
            model_name="logistic_regression",
            split_name="validation",
        ),
        make_prediction_frame(
            splits["test"],
            y_test,
            logistic_test_prob,
            ZERO_SHOT_MAJOR_TOPICS,
            logistic_thresholds,
            model_name="logistic_regression",
            split_name="test",
        ),
        make_prediction_frame(
            splits["validation"],
            y_validation,
            mlp_validation_prob,
            ZERO_SHOT_MAJOR_TOPICS,
            mlp_thresholds,
            model_name="mlp",
            split_name="validation",
        ),
        make_prediction_frame(
            splits["test"],
            y_test,
            mlp_test_prob,
            ZERO_SHOT_MAJOR_TOPICS,
            mlp_thresholds,
            model_name="mlp",
            split_name="test",
        ),
    ],
    ignore_index=True,
)

display(overall_metrics.sort_values(["Split", "Macro_F1", "Micro_F1"], ascending=[True, False, False]))
display(per_topic_metrics[per_topic_metrics["Split"] == "test"].sort_values(["Model", "F1"], ascending=[True, False]))


In [ ]:
# Optional grouped hotel CV for the linear baseline.
if RUN_GROUPED_CV:
    logistic_cv = run_grouped_logistic_cv(
        dataset,
        tag_column=FEATURE_SPECS[ACTIVE_FEATURE_SET],
        min_df=MIN_TAG_DF,
        n_splits=5,
    )
    display(logistic_cv)
    display(logistic_cv[["Macro_F1", "Micro_F1", "Top1_Accuracy", "Top2_Hit_Rate"]].mean().to_frame("mean"))
else:
    logistic_cv = pd.DataFrame()
    print("Grouped logistic CV skipped. Set RUN_GROUPED_CV = True to run it.")


In [ ]:
# Save tables, plots, thresholds, and the trained MLP state dict.
loss_fig, loss_ax = plt.subplots(figsize=(8, 4.5))
loss_ax.plot(mlp_history["epoch"], mlp_history["train_loss"], label="Train Loss")
loss_ax.plot(mlp_history["epoch"], mlp_history["validation_loss"], label="Validation Loss")
loss_ax.set_title("MLP training history")
loss_ax.set_xlabel("Epoch")
loss_ax.set_ylabel("BCE loss")
loss_ax.legend()
loss_fig.tight_layout()
loss_fig.savefig(OUTPUT_DIR / "topic_nn_review_level_loss_curve.png", dpi=200, bbox_inches="tight")
plt.show()

topic_plot_df = per_topic_metrics[per_topic_metrics["Split"] == "test"].copy()
topic_plot_df = topic_plot_df[topic_plot_df["Model"].isin(["logistic_regression", "mlp"])]
topic_fig, topic_ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=topic_plot_df, x="F1", y="Topic", hue="Model", ax=topic_ax)
topic_ax.set_title("Per-topic F1 on the grouped-hotel test split")
topic_ax.set_xlabel("F1")
topic_ax.set_ylabel("")
topic_fig.tight_layout()
topic_fig.savefig(OUTPUT_DIR / "topic_nn_review_level_test_per_topic_f1.png", dpi=200, bbox_inches="tight")
plt.show()

dataset_summary.to_csv(OUTPUT_DIR / "dataset_summary.csv", index=False)
split_summary.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)
feature_summary.to_csv(OUTPUT_DIR / "feature_set_summary.csv", index=False)
ablation_overall.to_csv(OUTPUT_DIR / "logistic_feature_ablation.csv", index=False)
ablation_per_topic.to_csv(OUTPUT_DIR / "logistic_feature_ablation_per_topic.csv", index=False)
tag_vocabulary.to_csv(OUTPUT_DIR / "tag_vocabulary.csv", index=False)
overall_metrics.to_csv(OUTPUT_DIR / "overall_metrics.csv", index=False)
per_topic_metrics.to_csv(OUTPUT_DIR / "per_topic_metrics.csv", index=False)
prediction_frames.to_csv(OUTPUT_DIR / "validation_test_predictions.csv", index=False)
mlp_history.to_csv(OUTPUT_DIR / "mlp_training_history.csv", index=False)
if not logistic_cv.empty:
    logistic_cv.to_csv(OUTPUT_DIR / "logistic_grouped_cv.csv", index=False)

save_json(
    OUTPUT_DIR / "config.json",
    {
        "random_state": RANDOM_STATE,
        "min_tag_df": MIN_TAG_DF,
        "active_feature_set": ACTIVE_FEATURE_SET,
        "feature_specs": FEATURE_SPECS,
        "topics": list(ZERO_SHOT_MAJOR_TOPICS),
        "split_rows": {name: int(len(df)) for name, df in splits.items()},
        "split_hotels": {name: int(df['Hotel_Name'].nunique()) for name, df in splits.items()},
        "active_feature_count": int(active_matrices['train'].shape[1]),
        "logistic_thresholds": {topic: float(threshold) for topic, threshold in zip(ZERO_SHOT_MAJOR_TOPICS, logistic_thresholds)},
        "mlp_thresholds": {topic: float(threshold) for topic, threshold in zip(ZERO_SHOT_MAJOR_TOPICS, mlp_thresholds)},
        "mlp_training": mlp_config,
        "run_grouped_cv": RUN_GROUPED_CV,
    },
)
torch.save(mlp_model.state_dict(), OUTPUT_DIR / "topic_nn_review_level_state_dict.pt")

print("Saved outputs:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name}")
